# Markowitz et al. (2023): dLight photometry claims, on dandiset 000559

Paper: Markowitz, Gillis, Jay et al. (2023), "Spontaneous behaviour is structured by reinforcement
without explicit reward", Nature 614, 108-117, doi:10.1038/s41586-022-05611-2. The claims file
(`claims/markowitz-2023.yaml`) was written from the version of record (Europe PMC, PMC9892006):
main text, legends, Methods and Extended Data legends.

Dataset: DANDI:000559, version 0.261113.1911 (doi:10.48324/dandi.000559/0.261113.1911). The paper
cites no version of the dandiset. This is the most recent published version; its release notes
say the earlier versions mislabelled the dLight-only mice and left the catch trials out, and the
claims file's description (groups, catch trials, session numbers) matches this version better.
The files are streamed; nothing is downloaded to disk.

This notebook tests the claims on dLight photometry (Figs. 1, 2, Extended Data Figs. 2, 3, 7).
The Opto-DA claims are in `optoda.ipynb`. Claims tested and blocked here:

| claim | here |
|---|---|
| ed2a-syllable-repertoire | tested |
| fig1e-velocity-timescales | tested |
| ed3c-keypoint-velocity | tested |
| fig1f-syllable-vs-movement-onset | tested |
| fig1f-dip-then-peak | tested |
| fig1h-syllable-dlight-differs | tested |
| fig1j-weak-kinematic-correlation | tested |
| fig1l-identity-not-decodable | blocked (too_large) |
| fig2a-food-reward-transients | blocked (data_missing) |
| fig2b-dlight-usage-across-syllables | tested |
| fig2i-dlight-entropy-across-syllables | tested |
| fig2b-across-experiments | tested |
| fig2e-dlight-predicts-future-usage | tested |
| fig2e-not-autocorrelation | tested |
| fig2g-usage-crosscorrelation | tested |
| fig2h-specific-to-syllable | tested |
| ed7-dms-fewer-transients, ed7b-dms-transient-shape, ed7d-dms-smaller-peaks | tested |
| fig3c-calibration, fig4i | blocked (data_missing) |
| others (encoding and decoding models, RL model, Fig. 4) | blocked, see the record |

In [ ]:
import zlib, time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import markowitz as M

PAPER = "markowitz-2023"
def rng_for(claim, k=0):
    return np.random.default_rng(zlib.crc32(f"{PAPER}/{claim}".encode()) + k)
def seed_of(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())

T0 = time.time()
A = M.list_assets()
print(A.kind.value_counts(dropna=False).to_string())
rows = [r for _, r in A[A.kind.isin(["photometry", "reinforcement-photometry"])].iterrows()]
SESS = M.load_many(M.load_photometry, rows)
print(len(SESS), "photometry files loaded in", round(time.time() - T0), "s")

In [ ]:
def first(v, default=np.nan):
    return v[0] if isinstance(v, list) and len(v) else default

S = pd.DataFrame([dict(i=i, path=d["path"], mouse=d["subject"], group=d["group"], ophys=d["has_ophys"],
                       sn=first(d.get("ts_session_number")), stim=first(d.get("ts_stim_duration")),
                       etype=first(d.get("ts_experiment_type"), ""), exclude=any(d.get("ts_exclude", [False])),
                       targets=[t for t in d.get("ts_target_syllable", []) if t >= 0],
                       maxpct=d.get("qc_max_pct", np.nan), refcorr=d.get("qc_corr", np.nan),
                       lab_max=first(d.get("ts_signal_max")), lab_corr=first(d.get("ts_signal_reference_corr")),
                       n=len(d["t"]) if d["t"] is not None else 0, start=d["start"])
                  for i, d in enumerate(SESS)])
S["area"] = np.where(S.group == "dms-dlight", "DMS", "DLS")
# quality control, reading (a): the Methods' maximum % dF/F0 > 1.5 and dLight-reference r < 0.6
S["qc"] = S.ophys & (S.maxpct > 1.5) & (S.refcorr < 0.6)
# baseline photometry experiments: no stimulation. dLight-only mice: all; dLight+ChrimsonR mice:
# session numbers -1 and 0 (code reading, which drops the post days 3 and 4), and the Methods
# reading, which adds the post days with the target syllable's instances removed.
chr_ = S.group == "dls-chrimson-dlight"
S["base_code"] = (~chr_ | S.sn.isin([-1, 0])) & ~S.exclude & (S.etype != "excitation_photometry")
S["base_methods"] = S.base_code | (chr_ & S.sn.isin([3, 4]) & ~S.exclude)
DLS = S[(S.area == "DLS") & S.qc & S.base_code]
DLSm = S[(S.area == "DLS") & S.qc & S.base_methods]
DMS = S[(S.area == "DMS") & S.qc]

In [ ]:
print("files by group, with and without photometry:")
print(S.groupby(["group", "ophys"]).size().to_string())
print("\nmice with photometry per group:", S[S.ophys].groupby("group").mouse.nunique().to_dict())
print("mice without any photometry file:", sorted(set(S.mouse) - set(S[S.ophys].mouse)))
print("\nQC (a), files failing:", S[S.ophys & ~S.qc][["path", "maxpct", "refcorr"]].to_string())
print("max % dF/F range", S.maxpct.min().round(2), S.maxpct.max().round(2),
      "; dLight-reference r range", S.refcorr.min().round(3), S.refcorr.max().round(3))
print("lab's signal_max against our max % dF/F: Spearman r =",
      round(stats.spearmanr(S.lab_max, S.maxpct, nan_policy="omit")[0], 3))
print("\nDLS baseline experiments (code reading):", len(DLS), "in", DLS.mouse.nunique(), "mice;",
      "with post days (Methods reading):", len(DLSm))
print(DLS.groupby("mouse").size().to_string())
print("\nDMS experiments:", len(DMS), "in", DMS.mouse.nunique(), "mice")
print(DMS.groupby("mouse").size().to_string())
d0 = SESS[DLS.i.iloc[0]]
fs = np.array([np.median(np.diff(SESS[i]["t"])) for i in S[S.ophys].i])
print("\nphotometry sample interval (s): min", fs.min().round(5), "max", fs.max().round(5))
print("syllable series sorted in time in every file:", all(d["syl_sorted"] for d in SESS))
print("samples per experiment: min", S[S.ophys].n.min(), "median", S[S.ophys].n.median(), "max", S[S.ophys].n.max(),
      "(30 min at 30 Hz is 54000)")
allsyl = np.concatenate([SESS[i]["syl"] for i in DLS.i])
print("offline syllable labels range", allsyl.min(), allsyl.max(), "; distinct", len(np.unique(allsyl)))
v = np.concatenate([SESS[i]["speed"] for i in DLS.i])
print("2D speed (mm/s) percentiles 1, 50, 99:", np.percentile(v, [1, 50, 99]).round(1))
zz = np.concatenate([SESS[i]["z"] for i in DLS.i])
print("processed dLight z: mean", np.nanmean(zz).round(3), "sd", np.nanstd(zz).round(3), "NaN fraction", np.isnan(zz).mean().round(4))
del allsyl, v, zz
# Clock check: syllable-associated dLight differs across syllables when measured at the syllable
# onsets; with the labels shifted in time the contrast should fall (one experiment per mouse).
for sh in [-60, -15, -6, 0, 6, 15, 60]:
    vals = []
    for m, sub in DLS.groupby("mouse"):
        d = SESS[sub.i.iloc[0]]
        s_, l_ = M.runs(d["syl"])
        s2 = np.clip(s_ + sh, 0, len(d["z"]) - 10)
        da = d["z"][s2[:, None] + np.arange(10)].max(1)
        vals.append(pd.DataFrame(dict(label=l_, da=da)))
    x = pd.concat(vals)
    H = stats.kruskal(*[g.da.to_numpy() for _, g in x.groupby("label") if len(g) > 20])[0]
    print(f"syllable labels shifted by {sh:+d} samples: Kruskal-Wallis H of instance dLight across syllables = {H:.0f}")

In [ ]:
# Syllable instances with their syllable-associated dLight (max z from onset to 0.3 s).
def inst_table(sub):
    out = []
    for _, r in sub.iterrows():
        x = M.instances(SESS[r.i])
        x["exp"] = r.i
        x["mouse"] = r.mouse
        x["post"] = r.sn in (3, 4)
        x["is_target"] = x.label.isin(r.targets) if r.sn in (3, 4) else False
        out.append(x)
    return pd.concat(out, ignore_index=True)

I = inst_table(DLSm)
I = I[~I.is_target]                    # Methods reading: target instances removed on post days
Ic = I[~I.post]                        # code reading: post days dropped
print("DLS instances, Methods reading:", len(I), "; code reading:", len(Ic))

## ed2a-syllable-repertoire
"MoSeq identifies 37 commonly used syllables, with median duration 400 ms (mean 566 ms, s.d. 636 ms)."

In [ ]:
# Reading (a): all DLS photometry experiments of the 16 dLight mice, with and without photometry,
# offline labels. Usage of a syllable is its share of a mouse's instances, averaged over mice.
def all_dls_instances():
    sub = S[(S.area == "DLS") & S.base_methods]
    rows_ = []
    for _, r in sub.iterrows():
        d = SESS[r.i]
        s, lab = M.runs(d["syl"])
        on = d["t"][s]
        rows_.append(pd.DataFrame(dict(mouse=r.mouse, label=lab.astype(int), dur=np.r_[np.diff(on), np.nan])))
    return pd.concat(rows_, ignore_index=True), sub

ALL, sub_a = all_dls_instances()
use = ALL.groupby("mouse").label.value_counts(normalize=True).unstack(fill_value=0)
mean_use = use.mean(0).sort_values(ascending=False)
SYL37 = sorted(mean_use[mean_use > 0.01].index)
print("reading (a):", len(sub_a), "experiments,", ALL.mouse.nunique(), "mice,", len(ALL), "instances")
print("syllables with mean usage > 1%:", len(SYL37))
dur = ALL.dur.dropna() * 1000
print(f"durations (ms): median {dur.median():.0f}, mean {dur.mean():.0f}, s.d. {dur.std():.0f}, n = {len(dur)}")
dur37 = ALL[ALL.label.isin(SYL37)].dur.dropna() * 1000
print(f"  of the {len(SYL37)} syllables only: median {dur37.median():.0f}, mean {dur37.mean():.0f}, s.d. {dur37.std():.0f}")
fu = pd.concat([pd.Series(SESS[i]["syl"]).value_counts(normalize=True).rename(SESS[i]["subject"]) for i in sub_a.i], axis=1)
print("variant, usage as share of frames:", (fu.T.groupby(level=0).mean().mean(0) > 0.01).sum(), "syllables > 1%")
print("variant, the 14 photometry mice only:", (use.loc[DLS.mouse.unique()].mean(0) > 0.01).sum(), "syllables > 1%")
print("syllables in use per mouse (any instance):", use.gt(0).sum(1).to_dict())

**Result: reproduced.** Under reading (a), the DLS photometry experiments of the 16 dLight mice, 36 syllables have a mean usage above 1% (37 within its tolerance of 3.7), and the 671,828 durations have a median of 400 ms, a mean of 561 ms and an s.d. of 627 ms, all within tolerance. Over the 14 photometry mice alone 37 syllables pass the 1% bar. Reading (b) is in `optoda.ipynb`.

Reading (b), all open-field experiments of the dandiset with offline labels, includes the Opto-DA
experiments; it is computed in `optoda.ipynb` (section ed2a), which loads those files.

## fig1h-syllable-dlight-differs
"Average syllable-associated dLight amplitudes differ across syllables (Kruskal-Wallis H = 209.29,
P < 1e-25, n = 518 mouse-syllable pairs)."

In [ ]:
# Onset-aligned waveforms per mouse, each z-scored over -10 to +10 s (Methods, dLight average
# waveform z-scoring), with circular-shuffle normalisation of the averages.
L = 601                      # -10 .. +10 s at 30 Hz
LAG = np.arange(-300, 301)

def snippets(i, starts):
    z = SESS[i]["z"]
    n = len(z)
    ok = (starts - 300 >= 0) & (starts + 300 < n)
    idx = starts[ok][:, None] + LAG[None, :]
    X = z[idx].astype(np.float32)
    good = np.all(np.isfinite(X), 1)
    X = X[good]
    X = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)
    return X, np.flatnonzero(ok)[good]

def shuffle_norm(X, groups, ngroups, cols, nshuf, rng):
    """Group averages of the rows of X at columns `cols`, z-scored against `nshuf` averages
    of circularly permuted rows (Methods, dLight average waveform z-scoring)."""
    from scipy.sparse import csr_matrix
    N = len(X)
    G = csr_matrix((np.ones(N), (groups, np.arange(N))), shape=(ngroups, N))
    cnt = np.asarray(G.sum(1)).ravel()
    cnt[cnt == 0] = np.nan
    avg = (G @ X[:, cols]) / cnt[:, None]
    sh = np.empty((nshuf, ngroups, len(cols)), np.float32)
    rows_ = np.arange(N)[:, None]
    for s in range(nshuf):
        k = rng.integers(0, X.shape[1], N)
        sh[s] = (G @ X[rows_, (cols[None, :] + k[:, None]) % X.shape[1]]) / cnt[:, None]
    mu, sd = sh.mean(0), sh.std(0)
    return avg, (avg - mu) / sd, (sh - mu) / sd

COLS = np.arange(300 - 30, 300 + 31)          # -1 .. +1 s
t03 = np.arange(30, 40)                        # columns of COLS for 0 .. 0.3 s
NSHUF = 1000

In [ ]:
# The 37 syllables of the dLight analyses: mean usage over the 14 photometry mice above 1%
# (all_behavior_00 keeps syllables whose mean usage over mice exceeds 1%; ed2a counts over all 16).
use14 = use.loc[sorted(DLS.mouse.unique())].mean(0)
SYL37 = sorted(use14[use14 > 0.01].index)
print("syllables with mean usage > 1% over the 14 photometry mice:", len(SYL37))

def movement_onsets(i):
    """Methods, Movement initiation analyses: centroid speed z-scored per experiment, 50-point
    boxcar, derivative cubed, scipy find_peaks with height, width and prominence 1."""
    from scipy.signal import find_peaks
    v = SESS[i]["speed"].astype(float)
    v = (v - v.mean()) / v.std()
    v = np.convolve(v, np.ones(50) / 50, mode="same")
    # derivative per second (z per s); per frame no peak reaches the height of 1 (an assumption)
    sc = (np.gradient(v) * M.FS) ** 3
    pk, _ = find_peaks(sc, height=1, width=1, prominence=1)
    return pk

def highpass(X):
    from scipy.signal import butter, filtfilt
    b, a = butter(2, 1.0 / (M.FS / 2), "high")        # order 2 is a standard choice
    Y = filtfilt(b, a, X, axis=1).astype(np.float32)
    return (Y - Y.mean(1, keepdims=True)) / Y.std(1, keepdims=True)

def instances_of(sub):
    return Ic if sub is DLS else DMSI

DMSI = inst_table(DMS)
t0 = time.time()
WAVE, SYL_SNIP_N, ONSET = {}, {}, {}
rng = rng_for("fig1f-syllable-vs-movement-onset")
rng_h = rng_for("fig1h-syllable-dlight-differs")
for area, sub_area in [("DLS", DLS), ("DMS", DMS)]:
    II = instances_of(sub_area)
    for m, sub in sub_area.groupby("mouse"):
        Xs, gs, Xm = [], [], []
        for i in sub.i:
            x = II[II.exp == i]
            X, keep = snippets(i, x.start.to_numpy())
            Xs.append(X)
            lab = x.label.to_numpy()[keep]
            gs.append(np.where(np.isin(lab, SYL37), np.searchsorted(SYL37, lab), len(SYL37)))
            Xm.append(snippets(i, movement_onsets(i))[0])
        X = np.concatenate(Xs); g = np.concatenate(gs); Xm = np.concatenate(Xm)
        out = {}
        # all syllable onsets, and movement initiations: shuffle-normalised averages
        _, out["syl"], out["syl_sh"] = shuffle_norm(X, np.zeros(len(X), int), 1, COLS, NSHUF, rng)
        _, out["mov"], out["mov_sh"] = shuffle_norm(Xm, np.zeros(len(Xm), int), 1, COLS, NSHUF, rng)
        # reading (a) of fig1f (DLS only): each snippet high-passed at 1 Hz, as the fig_1f, s5f
        # notebook; 100 shuffles (the Fig. 1f legend's number), to keep the run within its time
        if area == "DLS":
            _, out["syl_hp"], _ = shuffle_norm(highpass(X), np.zeros(len(X), int), 1, COLS, 100, rng)
            _, out["mov_hp"], _ = shuffle_norm(highpass(Xm), np.zeros(len(Xm), int), 1, COLS, 100, rng)
        out["n_syl"], out["n_mov"] = len(X), len(Xm)
        ONSET[(area, m)] = out
        if area == "DLS":
            sel = g < len(SYL37)
            avg, zn, _ = shuffle_norm(X[sel], g[sel], len(SYL37), COLS, NSHUF, rng_h)
            WAVE[m] = (avg, zn)
            SYL_SNIP_N[m] = np.bincount(g[sel], minlength=len(SYL37))
        del X, Xm
print("onset-aligned averages for", len(ONSET), "mice,", NSHUF, "shuffles each, in", round(time.time() - t0), "s")

In [ ]:
rows_ = []
for m, sub in DLS.groupby("mouse"):
    x = Ic[(Ic.mouse == m) & Ic.label.isin(SYL37)]
    a = x.groupby("label").da.mean().reindex(SYL37)
    avg, zn = WAVE[m]
    for j, s in enumerate(SYL37):
        rows_.append(dict(mouse=m, label=s, a=a[s], b=avg[j, t03].mean(), c=avg[j, t03].max(), d=zn[j, t03].max(),
                          n=SYL_SNIP_N[m][j]))
P1h = pd.DataFrame(rows_)
print("mouse-syllable pairs:", len(P1h), "; pairs with no instance:", P1h.a.isna().sum(),
      "; instances per pair: min", P1h.n.min(), "median", P1h.n.median())
for k, name in [("a", "mean syllable-associated dLight"), ("b", "mean of z-scored average, 0-0.3 s"),
                ("c", "max of z-scored average, 0-0.3 s"), ("d", "peak of shuffle-normalised average, 0-0.3 s")]:
    grp = [g[k].dropna().to_numpy() for _, g in P1h.groupby("label")]
    H, p = stats.kruskal(*grp)
    print(f"({k}) {name}: H = {H:.2f}, P = {p:.2e}, range of values {P1h[k].min():.2f} to {P1h[k].max():.2f}")
# Methods reading of baseline (post days with the target removed)
grp = [g.da.to_numpy() for _, g in I[I.label.isin(SYL37)].groupby(["mouse", "label"]).da.mean().reset_index().groupby("label")]
print("(a) with post days, target removed: H = %.2f, P = %.2e" % stats.kruskal(*grp))

**Result: partially reproduced.** Across the 518 mouse-syllable pairs (14 mice x 37 syllables) the Kruskal-Wallis test gives P below 1e-25 under every reading (H from 274.52 to 289.27), but H is above the tolerance of 209.29 +- 20.93 under all four readings.

## fig1f-syllable-vs-movement-onset
"The average syllable-associated dLight transient exceeds that associated with movement
initiation (P = 0.0006, n = 14 mice)."

In [ ]:
iw = np.arange(len(COLS))              # -1 .. +1 s
rows_ = []
for (area, m), o in ONSET.items():
    if area != "DLS":
        continue
    rows_.append(dict(mouse=m, n_syl=o["n_syl"], n_mov=o["n_mov"],
                      a_syl=np.abs(o["syl_hp"][0]).mean(), a_mov=np.abs(o["mov_hp"][0]).mean(),
                      b_syl=o["syl"][0, t03].max(), b_mov=o["mov"][0, t03].max(),
                      b100_syl=None, b100_mov=None))
F1 = pd.DataFrame(rows_)
# the legend's 100 shuffles: the same averages z-scored against the first 100 shuffles
for k in ["syl", "mov"]:
    vals = []
    for m in F1.mouse:
        o = ONSET[("DLS", m)]
        sh = o[k + "_sh"][:100]                          # already z-scored against all 1,000
        vals.append(((o[k][0, t03] - sh.mean(0)[0, t03]) / sh.std(0)[0, t03]).max())
    F1[f"b100_{k}"] = vals
print(F1.round(2).to_string())
for r in ["a", "b", "b100"]:
    w = stats.wilcoxon(F1[f"{r}_syl"], F1[f"{r}_mov"])
    print(f"reading ({r}): syllable > movement in {(F1[f'{r}_syl'] > F1[f'{r}_mov']).sum()} of {len(F1)} mice; "
          f"median {F1[f'{r}_syl'].median():.2f} against {F1[f'{r}_mov'].median():.2f}; Wilcoxon W = {w.statistic:.0f}, P = {w.pvalue:.2g}")

**Result: reproduced, under reading (b) and resting on an assumption.** The peak of the shuffle-normalised average from 0 to 0.3 s is higher for syllable onsets than for movement initiations in 14 of 14 mice (Wilcoxon P = 0.00012, with 1,000 or the first 100 shuffles). Under reading (a), the mean absolute z from -1 to +1 s of high-passed snippets, movement initiations are higher (4 of 14 mice favour syllables, P = 0.011). Movement initiations use the changepoint score's derivative per second, an assumption: with the derivative per frame no peak reaches the height of 1.

## fig1f-dip-then-peak
"DLS dopamine fluctuates at syllable transitions with a dip followed by a peak."

In [ ]:
tt = LAG[COLS] / M.FS
for key, label in [("syl", "1,000 shuffles")]:
    mice = sorted(F1.mouse)
    avg = np.mean([ONSET[("DLS", m)][key][0] for m in mice], 0)
    sh = np.mean([ONSET[("DLS", m)][key + "_sh"][:, 0, :] for m in mice], 0)   # shuffles averaged over mice
    lo, hi = np.percentile(sh, [2.5, 97.5], axis=0)
    below = np.flatnonzero(avg < lo); above = np.flatnonzero(avg > hi)
    imin, imax = np.argmin(avg), np.argmax(avg)
    print(f"{label}: minimum {avg[imin]:.2f} at {tt[imin]:+.3f} s (interval {lo[imin]:.2f} to {hi[imin]:.2f}); "
          f"maximum {avg[imax]:.2f} at {tt[imax]:+.3f} s (interval {lo[imax]:.2f} to {hi[imax]:.2f})")
    print("  times below the interval:", np.round(tt[below], 3))
    print("  times above the interval:", np.round(tt[above], 3))
    holds = len(below) > 0 and len(above) > 0 and tt[imin] < tt[imax] and avg[imin] < lo[imin] and avg[imax] > hi[imax]
    print("  dip below the interval followed by a peak above it:", holds)
    per_mouse = [(tt[np.argmin(ONSET[("DLS", m)][key][0])] < tt[np.argmax(ONSET[("DLS", m)][key][0])]) for m in mice]
    print("  mice whose minimum precedes their maximum within -1 to +1 s:", sum(per_mouse), "of", len(mice))
fig, ax = plt.subplots(figsize=(5, 3))
ax.fill_between(tt, lo, hi, color="0.85", label="95% shuffle interval")
ax.plot(tt, avg, "k", label="mean of 14 mice")
ax.axvline(0, ls=":", c="0.5"); ax.set_xlabel("time from syllable onset (s)"); ax.set_ylabel("dLight (z vs shuffles)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()
print("plotted range of the average:", avg.min().round(2), "to", avg.max().round(2))

**Result: reproduced, resting on the claims file's assumed window.** The mean of the 14 mice falls below the 95% shuffle interval from -0.93 to +0.2 s, with its minimum (-15.38) at the onset, and rises above it from +0.27 s, with its maximum (1.75) at +0.33 s; the minimum precedes the maximum in 13 of 14 mice.

## fig1j-weak-kinematic-correlation
"Syllable-associated dopamine transient amplitudes correlate only weakly with the velocity and
angular velocity of the syllable instance."

In [ ]:
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import KFold

def huber_cv_r(x, y, rng, n_rep=100, eps=1.05, fit_intercept=True):
    """Pearson r of pooled held-out predictions of a Huber regression, per repeat of fivefold CV."""
    x = np.asarray(x, float).reshape(len(x), -1); y = np.asarray(y, float)
    rs = []
    for _ in range(n_rep):
        pred = np.empty_like(y)
        for tr, te in KFold(5, shuffle=True, random_state=int(rng.integers(2**31))).split(x):
            h = HuberRegressor(epsilon=eps, fit_intercept=fit_intercept, max_iter=500).fit(x[tr], y[tr])
            pred[te] = h.predict(x[te])
        rs.append(np.corrcoef(pred, y)[0, 1])
    return np.array(rs)

def huber_shuffle(x, y, rng, n_shuf=1000, n_rep=5, **kw):
    """Null of held-out r with x permuted (fivefold repeated n_rep times, as the code's null)."""
    return np.array([np.nanmean(huber_cv_r(rng.permutation(x), y, rng, n_rep=n_rep, **kw)) for _ in range(n_shuf)])

In [ ]:
rng = rng_for("fig1j-weak-kinematic-correlation")
samp = pd.concat([g.sample(2000, random_state=int(rng.integers(2**31)))
                  for _, g in Ic[np.isfinite(Ic.dur)].groupby("mouse")])
print("instances:", len(samp), "from", samp.mouse.nunique(), "mice")
R1j = {}
for dl, dname in [("da", "(a) max z 0-0.3 s"), ("zmean", "(b) mean z over the instance")]:
    for kin in ["vel", "angvel"]:
        x = samp[kin].to_numpy(); y = samp[dl].to_numpy()
        if kin == "angvel":
            x = np.abs(x)
        # 5 repeats of fivefold CV (the claims file says 100; the spread over repeats is in the
        # third decimal); the shuffle P uses the plain Pearson r over 1,000 permutations, as the
        # Huber fits on 28,000 points are too slow to repeat 1,000 times here
        r = huber_cv_r(x, y, rng, n_rep=5)
        r0 = np.corrcoef(x, y)[0, 1]
        null = np.array([np.corrcoef(rng.permutation(x), y)[0, 1] for _ in range(1000)])
        p = (1 + np.sum(np.abs(null) >= abs(r0))) / 1001
        R1j[(dl, kin)] = r.mean()
        print(f"{dname}, {kin}: held-out r = {r.mean():.3f} (range over repeats {r.min():.3f} to {r.max():.3f}); "
              f"raw Pearson r = {r0:.3f}; shuffle |r| 95th pct {np.percentile(np.abs(null), 95):.3f}, P = {p:.4f}")
print("all |r| < 0.3 under reading (a):", all(abs(R1j[("da", k)]) < 0.3 for k in ["vel", "angvel"]),
      "; under (b):", all(abs(R1j[("zmean", k)]) < 0.3 for k in ["vel", "angvel"]))

**Result: reproduced, against the claims file's assumed bound of |r| < 0.3.** On 28,000 instances the held-out r is 0.074 (velocity) and 0.018 (angular velocity) under reading (a) and 0.060 and 0.006 under reading (b); the raw correlations are negative (-0.075 and -0.022 under (a)).

## fig2b-dlight-usage-across-syllables
"Syllables associated on average with more DLS dopamine in an experiment tend to be used more."

In [ ]:
def per_exp_syllable(II):
    """Per experiment and syllable: mean syllable-associated dLight, count, outgoing-transition entropy."""
    g = II.groupby(["mouse", "exp", "label"])
    T = g.agg(da=("da", "mean"), count=("da", "size")).reset_index()
    # outgoing transitions within the experiment, destinations among the 37 syllables, prior 1/36
    # on each (dm_entropy in the fig_2b, 2i notebook), log base 2
    ent = []
    for e, x in II.groupby("exp"):
        lab = x.label.to_numpy()
        src, dst = lab[:-1], lab[1:]
        ok = np.isin(dst, SYL37)
        C = np.zeros((lab.max() + 1, len(SYL37)))
        np.add.at(C, (src[ok], np.searchsorted(SYL37, dst[ok])), 1)
        H = M.dm_entropy(C, 1 / 36)
        ent.append(pd.DataFrame(dict(exp=e, label=np.arange(len(C)), entropy=H)))
    return T.merge(pd.concat(ent), on=["exp", "label"], how="left")

PE = per_exp_syllable(Ic)
PE["da_z"] = PE.groupby("exp").da.transform(lambda v: (v - v.mean()) / v.std())
tot = PE.groupby(["mouse", "label"])["count"].sum()
PM = PE.groupby(["mouse", "label"]).agg(da_z=("da_z", "mean"), count=("count", "mean"), entropy=("entropy", "mean"))
PM = PM[tot.reindex(PM.index) >= 100].reset_index()
print("mouse-syllable pairs with at least 100 instances:", len(PM), "in", PM.mouse.nunique(), "mice")

In [ ]:
def fig2_test(x, y, claim, n_shuf=1000):
    rng = rng_for(claim)
    r = huber_cv_r(x, y, rng, n_rep=100)
    null = huber_shuffle(x, y, rng, n_shuf=n_shuf, n_rep=5)
    p = (1 + np.sum(null >= r.mean())) / (1 + n_shuf)
    return r.mean(), r.min(), r.max(), p, null

r, rmin, rmax, p, null = fig2_test(PM.da_z, PM["count"], "fig2b-dlight-usage-across-syllables")
print(f"held-out r = {r:.3f} (repeats {rmin:.3f} to {rmax:.3f}); null 95th percentile {np.percentile(null, 95):.3f}; one-sided P = {p:.4f}; n = {len(PM)}")
print("raw Pearson r:", round(np.corrcoef(PM.da_z, PM["count"])[0, 1], 3), "; Spearman:", round(stats.spearmanr(PM.da_z, PM["count"])[0], 3))
print("per mouse Spearman r:", PM.groupby("mouse").apply(lambda g: round(stats.spearmanr(g.da_z, g["count"])[0], 2), include_groups=False).to_dict())
# code's variant: dLight z-scored within mouse across experiment-syllable rows, instances shorter than 0.3 s left out
Iv = Ic[Ic.dur >= 0.3]
PEv = Iv.groupby(["mouse", "exp", "label"]).agg(da=("da", "mean"), count=("da", "size")).reset_index()
PEv["da_z"] = PEv.groupby("mouse").da.transform(lambda v: (v - v.mean()) / v.std())
PMv = PEv.groupby(["mouse", "label"]).agg(da_z=("da_z", "mean")).join(PM.set_index(["mouse", "label"])["count"], how="inner").reset_index()
rv = huber_cv_r(PMv.da_z, PMv["count"], rng_for("fig2b-dlight-usage-across-syllables", 1), n_rep=20)
print(f"variant (code): held-out r = {rv.mean():.3f}, n = {len(PMv)}")

**Result: reproduced.** Over 703 mouse-syllable pairs, the held-out r between experiment-normalised dLight and counts is 0.178 (one-sided shuffle P = 0.001); the code's variant gives 0.171.

## fig2i-dlight-entropy-across-syllables

In [ ]:
r, rmin, rmax, p, null = fig2_test(PM.da_z, PM.entropy, "fig2i-dlight-entropy-across-syllables")
print(f"held-out r = {r:.3f} (repeats {rmin:.3f} to {rmax:.3f}); null 95th percentile {np.percentile(null, 95):.3f}; one-sided P = {p:.4f}; n = {len(PM)}")
print("raw Pearson r:", round(np.corrcoef(PM.da_z, PM.entropy)[0, 1], 3))

**Result: reproduced.** The held-out r between experiment-normalised dLight and outgoing-transition entropy is 0.199 (one-sided shuffle P = 0.001, n = 703).

## fig2b-across-experiments

In [ ]:
PEa = PE.copy()
nexp = PEa.groupby("mouse").exp.nunique()
avg_per_sess = PEa.groupby(["mouse", "label"])["count"].sum() / nexp.reindex(PEa.groupby(["mouse", "label"])["count"].sum().index.get_level_values(0)).to_numpy()
keep = avg_per_sess[avg_per_sess >= 50].index
PEa = PEa.set_index(["mouse", "label"]).loc[keep].reset_index()
zs = lambda v: (v - v.mean()) / v.std() if v.std() > 0 else v * np.nan
PEa["da_zs"] = PEa.groupby(["mouse", "label"]).da.transform(zs)
PEa["count_zs"] = PEa.groupby(["mouse", "label"])["count"].transform(zs)
PX = PEa.groupby(["mouse", "exp"]).agg(da=("da_zs", "mean"), freq=("count_zs", "mean")).dropna().reset_index()
print("syllable-mouse pairs kept (>= 50 per session on average):", len(keep), "; experiments:", len(PX))
r, rmin, rmax, p, null = fig2_test(PX.da, PX.freq, "fig2b-across-experiments")
print(f"held-out r = {r:.3f} (repeats {rmin:.3f} to {rmax:.3f}); null 95th percentile {np.percentile(null, 95):.3f}; one-sided P = {p:.4f}")
print("raw Pearson r:", round(np.corrcoef(PX.da, PX.freq)[0, 1], 3))

**Result: reproduced.** Over 145 experiments (339 syllable-mouse pairs kept), the held-out r between z-scored dLight and z-scored frequency is 0.197 (one-sided shuffle P = 0.002).

## ed7-dms-fewer-transients, ed7b-dms-transient-shape, ed7d-dms-smaller-peaks

In [ ]:
def crossings(z, thr=1.0, hold=3, floor=0.25):
    """Upward crossings of thr, kept when the next `hold` samples stay at or above `floor`
    (get_ncrossings in the fig_s07 notebook, as the claims file cites it)."""
    up = np.flatnonzero((z[:-1] < thr) & (z[1:] >= thr)) + 1
    ok = [i for i in up if i + hold < len(z) and np.all(z[i + 1:i + 1 + hold] >= floor)]
    return np.array(ok, int)

def exp_z(i):
    z = SESS[i]["z"].astype(float)
    return (z - np.nanmean(z)) / np.nanstd(z)

TR = []
for area, sub in [("DLS", DLS), ("DMS", DMS)]:
    for _, r in sub.iterrows():
        z = exp_z(r.i)
        c = crossings(z)
        nb = len(z) // 9000                            # 300-s bins
        rate = [np.sum((c >= k * 9000) & (c < (k + 1) * 9000)) / 300 for k in range(nb)]
        snip = [z[j - 90:j + 91] for j in c if j - 90 >= 0 and j + 91 <= len(z)]
        TR.append(dict(area=area, mouse=r.mouse, exp=r.i, rate=np.mean(rate) if rate else np.nan, n=len(c),
                       wave=np.mean(snip, 0) if snip else np.full(181, np.nan)))
TR = pd.DataFrame(TR)
print(TR.groupby("area").agg(experiments=("exp", "size"), mice=("mouse", "nunique"), transients=("n", "sum")).to_string())

In [ ]:
pm = TR.groupby(["area", "mouse"]).rate.agg(["mean", "median"]).reset_index()
for stat_ in ["mean", "median"]:
    a = pm[pm.area == "DLS"][stat_]; b = pm[pm.area == "DMS"][stat_]
    u = stats.mannwhitneyu(a, b, alternative="two-sided")
    print(f"per-mouse {stat_} rate (per s): DLS median {a.median():.3f} (n={len(a)}), DMS median {b.median():.3f} (n={len(b)}); "
          f"U = {u.statistic:.0f}, f = {u.statistic / (len(a) * len(b)):.2f}, P = {u.pvalue:.4f}")
print(pm.round(3).to_string())

**Result: reproduced.** DMS mice have fewer transients per second than DLS mice (median of per-mouse means 0.822 against 0.885; U = 100, f = 0.89, P = 0.0016, 0.0065 after Holm), and U is within the tolerance of 97 +- 9.7.

In [ ]:
def shape(w):
    """fig_s07 notebook as cited: re-align where the slope first exceeds 0.035, then over 0-2 s area
    (trapezoid), time to peak, and full width at half maximum after cubic interpolation to 500 Hz."""
    from scipy.interpolate import interp1d
    sl = np.diff(w)
    k = np.flatnonzero(sl[:90] > 0.035)
    st = k[0] if len(k) else 90
    seg = w[st:st + 61]
    t = np.arange(len(seg)) / M.FS
    area_ = np.trapezoid(seg, t)
    tp = t[np.argmax(seg)]
    f = interp1d(t, seg, kind="cubic")
    tf = np.arange(0, t[-1], 1 / 500)
    sf = f(tf)
    half = sf.min() + (sf.max() - sf.min()) / 2
    above = np.flatnonzero(sf >= half)
    fwhm = (above[-1] - above[0]) / 500 if len(above) else np.nan
    shape.tp_interp = tf[np.argmax(sf)]               # variant: time to peak on the 500 Hz curve
    return area_, tp, fwhm

SH = []
for (area, m), g in TR.groupby(["area", "mouse"]):
    w = np.nanmean(np.stack(g.wave), 0)
    a_, tp, fw = shape(w)
    SH.append(dict(area=area, mouse=m, area_=a_, ttp=tp, fwhm=fw, ttp500=shape.tp_interp))
SH = pd.DataFrame(SH)
print(SH.round(3).to_string())
ps, us = [], []
for k in ["area_", "ttp", "fwhm"]:
    a = SH[SH.area == "DLS"][k]; b = SH[SH.area == "DMS"][k]
    u = stats.mannwhitneyu(a, b, alternative="two-sided"); ps.append(u.pvalue); us.append(u.statistic)
    print(f"{k}: DLS median {a.median():.3f}, DMS median {b.median():.3f}, U = {u.statistic:.0f}, P = {u.pvalue:.4f}")
a = SH[SH.area == "DLS"].ttp500; b = SH[SH.area == "DMS"].ttp500
u = stats.mannwhitneyu(a, b, alternative="two-sided")
print(f"variant, time to peak on the 500 Hz curve: DLS median {a.median():.3f}, DMS median {b.median():.3f}, U = {u.statistic:.0f}, P = {u.pvalue:.4f}")
rate_p = stats.mannwhitneyu(pm[pm.area == "DLS"]["mean"], pm[pm.area == "DMS"]["mean"]).pvalue
print("Holm over the four statistics (rate, area, time to peak, width):", np.round(M.holm([rate_p] + ps), 4))

**Result: not reproduced.** Width differs (U = 20, P = 0.0137, 0.0412 after Holm), but time to peak does not after Holm (U = 32, P = 0.0352, 0.0584 after Holm), and none of the three U values is within its tolerance (4, 16 and 27 in the paper). Area is not significant after Holm (P = 0.0292, 0.0584 after Holm), as in the paper.

In [ ]:
pk_cols = np.arange(30 - 5, 30 + 16)           # -5 .. +15 samples around onset, in COLS
PK = pd.DataFrame([dict(area=a, mouse=m, syl=o["syl"][0, pk_cols].max(), mov=o["mov"][0, pk_cols].max(),
                        n_syl=o["n_syl"], n_mov=o["n_mov"]) for (a, m), o in ONSET.items()])
print(PK.round(2).to_string())
for k in ["mov", "syl"]:
    a = PK[PK.area == "DLS"][k]; b = PK[PK.area == "DMS"][k]
    u = stats.mannwhitneyu(a, b, alternative="two-sided")
    print(f"{k}: DLS median {a.median():.2f}, DMS median {b.median():.2f}; U = {u.statistic:.0f} (max {len(a) * len(b)}), P = {u.pvalue:.2e}")
print("instances per mouse differ between areas and the shuffle z grows with them: syllable onsets per mouse, DLS median",
      PK[PK.area == "DLS"].n_syl.median(), "DMS median", PK[PK.area == "DMS"].n_syl.median())

**Result: not reproduced.** Peaks at syllable onsets (U = 77, P = 0.165) and at movement initiations (U = 51, P = 0.764) do not differ between the 14 DLS and 8 DMS mice; the paper reports U = 109 and 110.

## fig2e-dlight-predicts-future-usage, fig2e-not-autocorrelation, fig2h-specific-to-syllable
Methods, "Analysing the moment-to-moment relationship between dLight and syllable statistics
within an experiment", with the details the claims file takes from dlight_03, dlight_04 and dlight_08.

In [ ]:
SEG = np.array([0, 360, 660, 900, 1680, 1800.01])     # segment edges (s), Extended Data Fig. 2e
NS = np.arange(10, 391, 20)                           # n = 10 .. 390 syllables
OFFS = np.arange(-5, 6)
NMAX = NS.max()

def fig2e_table(II):
    """Per index instance (one of the 37 syllables, with 390 following syllables and 5 before in
    the experiment): dLight, group (mouse, syllable, segment), and the behavioural measures."""
    out_x, out_Y, out_g, out_exp, out_seg = [], [], [], [], []
    for e, x in II.groupby("exp", sort=False):
        lab = x.label.to_numpy(); da = x.da.to_numpy(); vel = x.vel.to_numpy()
        t = x.onset.to_numpy() - x.onset.to_numpy()[0]
        n = len(lab)
        j = np.arange(5, n - NMAX)
        j = j[np.isin(lab[j], SYL37)]
        if len(j) == 0:
            continue
        oh = np.zeros((n + 1, lab.max() + 1), np.int32)
        oh[np.arange(n) + 1, lab] = 1
        coh = np.cumsum(oh, 0)                              # coh[k, L] = count of L in lab[:k]
        cda = np.r_[0, np.cumsum(da)]
        cvel = np.r_[0, np.cumsum(vel)]
        Y = []
        for nn in NS:                                       # same syllable among the next n - 1
            Y.append(coh[j + nn, lab[j]] - coh[j + 1, lab[j]])
        for nn in NS:                                       # mean dLight of the next n - 1 instances
            Y.append((cda[j + nn] - cda[j + 1]) / (nn - 1))
        for nn in NS:                                       # mean velocity over instances 0 .. n - 1
            Y.append((cvel[j + nn] - cvel[j]) / nn)
        for o in OFFS:                                      # syllable at offset o, its count in 10 from there
            L2 = lab[j + o]
            Y.append(coh[j + o + 10, L2] - coh[j + o, L2])
        out_Y.append(np.stack(Y, 1).astype(float))
        out_x.append(da[j])
        seg = np.searchsorted(SEG, t[j], side="right") - 1
        out_seg.append(seg)
        out_exp.append(np.full(len(j), e))
        out_g.append(list(zip([x.mouse.iloc[0]] * len(j), lab[j], seg)))
    X = np.concatenate(out_x); Y = np.concatenate(out_Y)
    seg = np.concatenate(out_seg); exp = np.concatenate(out_exp)
    keys = pd.Series([f"{a}|{b}|{c}" for a, b, c in sum(out_g, [])])
    gid, uniq = pd.factorize(keys)
    gseg = np.array([int(u.split("|")[2]) for u in uniq])
    return X, Y, gid, gseg, seg, exp

def grouped_r(x, Y, G, Sy, Syy, N):
    Sx = G @ x; Sxx = G @ (x * x)
    Sxy = G @ (x[:, None] * Y)
    num = N[:, None] * Sxy - Sx[:, None] * Sy
    den = np.sqrt((N * Sxx - Sx ** 2)[:, None] * (N[:, None] * Syy - Sy ** 2))
    with np.errstate(invalid="ignore", divide="ignore"):
        return num / den

def seg_mean(R, gseg):
    return np.stack([np.nanmean(R[gseg == s], 0) for s in range(5)])      # 5 segments x K

def fig2e_run(II, nshuf, rng):
    from scipy.sparse import csr_matrix
    X, Y, gid, gseg, seg, exp = fig2e_table(II)
    G = csr_matrix((np.ones(len(X)), (gid, np.arange(len(X)))))
    N = np.asarray(G.sum(1)).ravel()
    Sy = G @ Y; Syy = G @ (Y * Y)
    obs = seg_mean(grouped_r(X, Y, G, Sy, Syy, N), gseg)
    blocks = pd.Series(np.arange(len(X))).groupby([exp, seg]).indices.values()
    sh = np.empty((nshuf,) + obs.shape)
    for s in range(nshuf):
        xp = X.copy()
        for b in blocks:
            xp[b] = X[rng.permutation(b)]
        sh[s] = seg_mean(grouped_r(xp, Y, G, Sy, Syy, N), gseg)
    mu, sd = np.nanmean(sh, 0), np.nanstd(sh, 0)
    zobs = np.nanmean((obs - mu) / sd, 0)                  # z per segment, averaged over segments
    zsh = np.nanmean((sh - mu) / sd, 1)
    return dict(X=X, Y=Y, groups=len(N), n=len(X), zobs=zobs, zsh=zsh, robs=np.nanmean(obs, 0))

t0 = time.time()
F2 = fig2e_run(Ic, NSHUF, rng_for("fig2e-dlight-predicts-future-usage"))
print("index instances:", F2["n"], "; mouse-syllable-segment groups:", F2["groups"], "; in", round(time.time() - t0), "s")
K = len(NS)
P_one = (1 + (F2["zsh"] >= F2["zobs"]).sum(0)) / (1 + NSHUF)
P_two = 2 * stats.norm.sf(np.abs((F2["zobs"] - F2["zsh"].mean(0)) / F2["zsh"].std(0)))   # normal approximation
band = np.percentile(np.abs(F2["zsh"]), 95, axis=0)

In [ ]:
u = slice(0, K)
tab = pd.DataFrame(dict(n=NS, mean_r=F2["robs"][u], z=F2["zobs"][u], P_one_sided=P_one[u],
                        P_two_sided_holm=M.holm(P_two[u])))
print(tab.round(4).to_string(index=False))
sig = tab.P_one_sided < 0.05
print("significant (one-sided P < 0.05) at n =", NS[sig.to_numpy()].tolist())
print("reading (a), every n up to 390:", bool(sig.all()), "; reading (b), every n up to 210:", bool(sig[NS <= 210].all()),
      "; at n = 10:", bool(sig.iloc[0]))
print("code variant (two-sided, Holm) significant at n =", NS[(tab.P_two_sided_holm < 0.05).to_numpy() & (tab.z > 0).to_numpy()].tolist())

**Result: reproduced.** The correlation between an instance's dLight and later use of the same syllable is significantly positive (one-sided P <= 0.002) at every n from 10 to 390 syllables (z from 6.72 at n = 10 to 1.76 at n = 390), and under the code's two-sided Holm-corrected test as well.

In [ ]:
ac = slice(K, 2 * K); ve = slice(2 * K, 3 * K)
tab2 = pd.DataFrame(dict(n=NS, z_usage=F2["zobs"][u], z_autocorr=F2["zobs"][ac], band_autocorr=band[ac],
                         z_velocity=F2["zobs"][ve], band_velocity=band[ve], usage_sig=P_one[u] < 0.05))
print(tab2.round(3).to_string(index=False))
inside = (np.abs(tab2.z_autocorr) <= tab2.band_autocorr) & (np.abs(tab2.z_velocity) <= tab2.band_velocity) & tab2.usage_sig
print("n at which both curves are inside the shuffle band while usage is significant:", NS[inside.to_numpy()].tolist())

**Result: partially reproduced.** Both curves drop sharply after n = 10 (autocorrelation z from 5.24 to -3.76 at n = 30; velocity from -10.61 to 0.95), while usage stays significant, but at no n are both inside the shuffle band: the autocorrelation stays negative and outside it (z from -1.38 to -4.24) from n = 30 on. Neither of the criterion's two cases applies as written.

In [ ]:
oo = slice(3 * K, 3 * K + len(OFFS))
P_o = (1 + (F2["zsh"][:, oo] >= F2["zobs"][oo]).sum(0)) / (1 + NSHUF)
tab3 = pd.DataFrame(dict(offset=OFFS, mean_r=F2["robs"][oo], z=F2["zobs"][oo], P_one_sided=P_o, P_holm=M.holm(P_o)))
print(tab3.round(4).to_string(index=False))
z0 = tab3.z[tab3.offset == 0].iloc[0]
others = tab3[tab3.offset != 0]
print("offset 0 significant (Holm):", bool(tab3.P_holm[tab3.offset == 0].iloc[0] < 0.05),
      "; other offsets significant:", others.offset[others.P_holm < 0.05].tolist(),
      "; every other z below half of offset 0's:", bool((others.z < z0 / 2).all()))

**Result: not reproduced.** Offset 0 is significant (z = 6.72, Holm P = 0.011), but so are offsets -1 (z = 4.34) and +1 (z = 5.17), whose z exceed half of offset 0's.

## fig1e-velocity-timescales

In [ ]:
BS = np.arange(0.25, 59.76, 0.5)
def fig1e_exp(i, thr):
    z = exp_z(i); v = SESS[i]["speed"].astype(float)
    t = np.arange(len(z)) / M.FS
    cross = np.r_[False, (z[:-1] < thr) & (z[1:] >= thr)].astype(float)
    out = []
    for bs in BS:
        b = np.floor(t / bs).astype(int)
        cnt = np.bincount(b)
        mode = np.bincount(cnt[cnt > 0]).argmax()
        keep = cnt[b] == mode
        bb = b[keep]
        n = np.bincount(bb)
        ok = n > 0
        out.append((np.bincount(bb, z[keep])[ok] / n[ok], np.bincount(bb, cross[keep])[ok] / bs,
                    np.bincount(bb, v[keep])[ok] / n[ok]))
    return out

def zs1(a):
    return (a - a.mean()) / a.std()

def fig1e_run(thr, rng, nshuf):
    mice = sorted(DLS.mouse.unique())
    R = {k: np.full((len(DLS), len(BS)), np.nan) for k in ["amp", "rate"]}
    SHR = {k: np.zeros((nshuf, len(mice), len(BS))) for k in ["amp", "rate"]}
    cnt = np.zeros(len(mice))
    for e, (_, r) in enumerate(DLS.iterrows()):
        mi = mice.index(r.mouse); cnt[mi] += 1
        for b, (amp, rate, v) in enumerate(fig1e_exp(r.i, thr)):
            zv = zs1(v)
            for k, x in [("amp", amp), ("rate", rate)]:
                if x.std() == 0 or len(x) < 3:
                    continue
                zx = zs1(x)
                R[k][e, b] = np.mean(zx * zv)
                P = rng.permuted(np.tile(zx, (nshuf, 1)), axis=1)
                SHR[k][:, mi, b] += P @ zv / len(zv)
    out = {}
    for k in ["amp", "rate"]:
        per_mouse = pd.DataFrame(R[k]).groupby(DLS.mouse.to_numpy()).mean()
        obs = per_mouse.mean(0).to_numpy()
        sh = (SHR[k] / cnt[None, :, None]).mean(1)
        out[k] = (obs, sh)
    return out

t0 = time.time()
FE = {thr: fig1e_run(thr, rng_for("fig1e-velocity-timescales", j), NSHUF) for j, thr in enumerate([1.96, 1.0])}
print("computed in", round(time.time() - t0), "s;", len(DLS), "experiments,", DLS.mouse.nunique(), "mice;", len(BS), "bin sizes")
def verdict(obs, sh):
    lo, hi = np.percentile(sh, [2.5, 97.5], axis=0)
    neg_a = obs < lo; pos_a = obs > hi
    # two-sided P from the z of the observed mean r against the shuffles (normal approximation: with
    # 1,000 shuffles an empirical P cannot pass Holm over 120 bin sizes)
    p2 = 2 * stats.norm.sf(np.abs((obs - sh.mean(0)) / sh.std(0)))
    ph = M.holm(p2) < 0.05
    run = np.zeros(len(ph), bool)                     # only runs of 5 or more consecutive bin sizes
    k = 0
    while k < len(ph):
        if ph[k]:
            j = k
            while j < len(ph) and ph[j]:
                j += 1
            if j - k >= 5:
                run[k:j] = True
            k = j
        else:
            k += 1
    short, long_ = BS < 10, BS >= 10
    res = {}
    for name, sig in [("(a) legend, P < 0.05 per bin size", None), ("(b) code, Holm, runs of 5", run)]:
        neg = neg_a if sig is None else (sig & (obs < 0))
        pos = pos_a if sig is None else (sig & (obs > 0))
        holds = bool(neg[short].any() and not pos[short].any() and pos[long_].any())
        res[name] = holds
        print(f"   {name}: significantly negative at {neg[short].sum()} of {short.sum()} bin sizes < 10 s, "
              f"positive at {pos[short].sum()} of them; significantly positive at {pos[long_].sum()} of {long_.sum()} bin sizes >= 10 s -> {holds}")
    return res
for thr in FE:
    for k in ["amp", "rate"]:
        obs, sh = FE[thr][k]
        if k == "amp" and thr == 1.0:
            continue
        print(f"{'amplitude' if k == 'amp' else f'transient rate, threshold {thr}'}: mean r at 0.25 s {obs[0]:.3f}, 5.25 s {obs[10]:.3f}, "
              f"9.75 s {obs[19]:.3f}, 20.25 s {obs[40]:.3f}, 59.75 s {obs[-1]:.3f}; sign change near {BS[np.flatnonzero(np.diff(np.sign(obs)))].round(2).tolist()} s")
        verdict(obs, sh)
fig, ax = plt.subplots(figsize=(5, 3))
for thr, k, lab_ in [(1.96, "amp", "amplitude"), (1.96, "rate", "rate (1.96)"), (1.0, "rate", "rate (1 s.d.)")]:
    ax.plot(BS, FE[thr][k][0], label=lab_)
ax.axhline(0, c="0.5", ls=":"); ax.set_xlabel("bin size (s)"); ax.set_ylabel("mean r with velocity"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Result: partially reproduced.** The transient rate (upward crossings of 1.96) holds under both readings: significantly negative at all 20 bin sizes below 10 s and significantly positive at 66 (legend's test) or 5 (code's test) of 100 bin sizes of 10 s or more. The binned amplitude is negative at every bin size (from -0.212 at 5.25 s to -0.053 at 59.75 s) and never significantly positive, as is the 1-s.d. rate.

## fig2g-usage-crosscorrelation

In [ ]:
def fig2g_run(step, rng, nshuf):
    """3-min windows (non-overlapping, or sliding by `step` s); per syllable and experiment mean
    syllable-associated dLight and count per window, correlated at lags; averaged over syllables and
    experiments; null from dLight permuted across windows within experiment and syllable."""
    W = 180.0
    lags_s = np.arange(-450, 451, 30) if step < W else np.arange(-2, 3) * 180.0   # within +-7.5 min
    obs_acc, sh_acc, n_acc = 0, 0, 0
    for e, x in Ic[Ic.label.isin(SYL37)].groupby("exp"):
        t = x.onset.to_numpy() - SESS[e]["t"][0]
        starts = np.arange(0, 1800 - W + 1e-9, step)
        for lab, g in x.groupby("label"):
            tt = g.onset.to_numpy() - SESS[e]["t"][0]; da = g.da.to_numpy()
            cnt = np.array([np.sum((tt >= s0) & (tt < s0 + W)) for s0 in starts])
            dm = np.array([da[(tt >= s0) & (tt < s0 + W)].mean() if c > 0 else np.nan for s0, c in zip(starts, cnt)])
            if np.isnan(dm).any() or dm.std() == 0 or cnt.std() == 0:
                continue
            P = np.vstack([dm, np.array([rng.permutation(dm) for _ in range(nshuf)])])
            row = []
            for L in lags_s:
                k = int(round(L / step))
                if k >= 0:
                    a, b = P[:, :len(dm) - k], cnt[k:]
                else:
                    a, b = P[:, -k:], cnt[:len(dm) + k]
                if a.shape[1] < 3 or b.std() == 0:
                    row.append(np.full(nshuf + 1, np.nan)); continue
                za = (a - a.mean(1, keepdims=True)) / a.std(1, keepdims=True)
                row.append(za @ ((b - b.mean()) / b.std()) / len(b))
            row = np.array(row).T                         # (1 + nshuf) x lags
            ok = np.isfinite(row)
            obs_acc = obs_acc + np.where(ok, row, 0)
            n_acc = n_acc + ok
    R_ = obs_acc / n_acc
    obs, sh = R_[0], R_[1:]
    z = (obs - sh.mean(0)) / sh.std(0)
    p = (1 + (sh >= obs).sum(0)) / (1 + len(sh))
    return lags_s, obs, z, p

for name, step, k in [("(a) non-overlapping 3-min windows", 180.0, 0), ("(b) windows sliding by 30 s", 30.0, 1)]:
    lags_s, obs, z, p = fig2g_run(step, rng_for("fig2g-usage-crosscorrelation", k), NSHUF)
    j = np.nanargmax(z)
    print(f"{name}: peak z {z[j]:.2f} at lag {lags_s[j] / 60:+.1f} min (mean r {obs[j]:.3f}), one-sided P = {p[j]:.4f}")
    print("   z by lag (min):", dict(zip((lags_s / 60).round(1).tolist(), np.round(z, 2).tolist())))
    pk = []
    for off in [1000, 2000, 3000, 4000]:
        _, _, z2, p2 = fig2g_run(step, rng_for("fig2g-usage-crosscorrelation", k + off), NSHUF)
        pk.append((round(float(np.nanmax(z2)), 2), round(float(p2[np.nanargmax(z2)]), 4)))
    print("   peak z and P under seeds +1000..+4000:", pk)

**Result: partially reproduced.** Under reading (b), windows sliding by 30 s, the peak z is 1.81 at +1.5 min with one-sided P = 0.039 (P < 0.05, not < 0.001); under reading (a), non-overlapping windows, the peak z is 1.29 at lag 0 (P = 0.10). The peak P under the four further seeds is printed above.

## ed3c-keypoint-velocity
"In keypoint-tracked mice, 2D velocity is negatively correlated with dLight in 400-ms bins
(r = -0.16), and forelimb velocity, with other kinematics partialled out, correlates only
negligibly (r = -0.02)."

In [ ]:
KP = M.load_many(M.load_keypoint, [r for _, r in A[A.kind == "keypoint"].iterrows()], 5)
rng = rng_for("ed3c-keypoint-velocity")
def vel2(p):                       # difference over 2 frames divided by 2 (Methods), in mm per frame
    v = np.full(len(p), np.nan)
    v[2:] = np.linalg.norm(p[2:] - p[:-2], axis=1) / 2
    return v * M.FS
B = []
for d in KP:
    spine = np.mean([d["rostral_spine"], d["medial_spine"], d["caudal_spine"]], 0)
    v2 = vel2(spine[:, :2]); v3 = vel2(spine); h = spine[:, 2]
    fl = np.mean([vel2(d["left_forepaw"]), vel2(d["right_forepaw"])], 0)
    n = min(len(d["z"]), len(v2)) // 12 * 12
    df = pd.DataFrame(dict(z=d["z"][:n], v2=v2[:n], v3=v3[:n], h=h[:n], fl=fl[:n])).groupby(np.arange(n) // 12).mean().dropna()
    X = np.c_[np.ones(len(df)), df.v2, df.v3, df.h]
    df["zpart"] = df.z - X @ np.linalg.lstsq(X, df.z, rcond=None)[0]
    df["mouse"] = d["subject"]
    B.append(df)
    print(f"{d['subject']}: {len(d['z'])} samples ({len(d['z']) / M.FS / 60:.1f} min), keypoint rate {d['rate']} Hz, "
          f"dLight-reference r {d['refcorr']:.3f}, 400-ms bins {len(df)}; 2D speed median {np.nanmedian(v2):.1f} mm/s")
B = pd.concat(B)
def shuf_p(x, y, rng, n=1000):
    r = np.corrcoef(x, y)[0, 1]
    null = np.array([np.corrcoef(rng.permutation(x), y)[0, 1] for _ in range(n)])
    return r, (1 + np.sum(null <= r)) / (1 + n)          # one-sided, negative direction
for name, a, b in [("2D velocity and dLight", "v2", "z"), ("forelimb velocity and partialled dLight", "fl", "zpart")]:
    r, p = shuf_p(B[a].to_numpy(), B[b].to_numpy(), rng)
    pm = B.groupby("mouse").apply(lambda g: np.corrcoef(g[a], g[b])[0, 1], include_groups=False)
    print(f"{name}: pooled r = {r:.3f} (one-sided shuffle P = {p:.4f}); per-mouse mean r = {pm.mean():.3f} "
          f"(per mouse {pm.round(3).to_dict()})")

**Result: partially reproduced.** In 400-ms bins of the 5 keypoint mice, 2D velocity and dLight correlate at r = -0.170 (pooled; per-mouse mean -0.169; one-sided shuffle P = 0.001), within -0.16 +- 0.016. The forelimb correlation with partialled dLight is -0.026 (pooled; per-mouse mean -0.027), which rounds to -0.03 and misses -0.02 +- 0.002.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| ed2a-syllable-repertoire | reproduced |
| fig1e-velocity-timescales | partially reproduced (rate yes, amplitude no) |
| ed3c-keypoint-velocity | partially reproduced (2D r yes, forelimb r -0.026) |
| fig1f-syllable-vs-movement-onset | reproduced under reading (b), assumption on movement onsets |
| fig1f-dip-then-peak | reproduced (assumed window) |
| fig1h-syllable-dlight-differs | partially reproduced (H above tolerance) |
| fig1j-weak-kinematic-correlation | reproduced (assumed bound) |
| fig2b-dlight-usage-across-syllables, fig2i-dlight-entropy-across-syllables, fig2b-across-experiments | reproduced |
| fig2e-dlight-predicts-future-usage | reproduced |
| fig2e-not-autocorrelation | partially reproduced |
| fig2g-usage-crosscorrelation | partially reproduced |
| fig2h-specific-to-syllable | not reproduced |
| ed7-dms-fewer-transients | reproduced |
| ed7b-dms-transient-shape, ed7d-dms-smaller-peaks | not reproduced |
| fig1l, fig2a, fig2k, ed6b, ed6e, ed7-dms-no-usage-prediction, ed7e, fig3c, Fig. 4, Fig. 5 | blocked (see the attempt record) |

How the analysis differs from the paper's: the DLS baseline set is the code's (stimulation
mice's post days dropped), 145 experiments where the paper counts 216 in Fig. 1b; the files' dF/F
is taken as the Methods' dF/F0; the 37 syllables are those above 1% mean usage over the 14
photometry mice; movement initiations take the changepoint derivative per second; fig1j uses 5
repeats of cross-validation and a Pearson shuffle null; fig2e drops index instances without 390
following syllables; seed ranges were computed only for fig2g.

What was found about the dataset: the two dLight-only DLS mice without photometry (dls-dlight-6
and -8) and two DMS mice without photometry (dms-dlight-7 and -9) match the paper's exclusions in
DLS but leave 10 DMS mice in the dandiset for the paper's 8; 22 DLS and 92 DMS photometry-session
files have behaviour but no photometry; one dLight+ChrimsonR file has a dF/F of up to 10,804% and a
dLight-reference correlation of 0.998 and fails quality control; the syllable-associated dLight
contrast peaks at the labels' own timing (H 888 at 0, 941 at -6 samples, 79 at +60), consistent
with a shared clock; the food-reward and calibration sessions are not in the dandiset.